<a href="https://colab.research.google.com/github/norahmunoz/ds2002-fa26/blob/main/2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [9]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [10]:
df['revenue'] = df['qty'] * df['price']

print("Total revenue:", df['revenue'].sum())
print("Total units:", df['qty'].sum())

print("There are", len(df), "orders.")
print("The total revenue is $8,520.00, and the total number of units sold is 783.")

Total revenue: 8520.0
Total units: 783
There are 400 orders.
The total revenue is $8,520.00, and the total number of units sold is 783.


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [11]:
by_category = df.groupby('category')['revenue'].sum().reset_index()

by_category['share_pct'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
)

by_category = by_category.sort_values('revenue', ascending=False)

print(by_category)

print("Food had the highest revenue, with $4,293.00, which was about 50.4% of total revenue.")

   category  revenue  share_pct
1      Food   4293.0  50.387324
2     Merch   1771.5  20.792254
0     Drink   1554.0  18.239437
3  RainGear    901.5  10.580986
Food had the highest revenue, with $4,293.00, which was about 50.4% of total revenue.


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [12]:
by_vendor = df.groupby('vendor_id').agg(
    avg_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
)

by_vendor = by_vendor.sort_values('avg_revenue', ascending=False)

print(by_vendor)

print("V-01 had the highest average order revenue at about $22.60 per order, based on 94 orders.")

           avg_revenue  order_count
vendor_id                          
V-01         22.595745           94
V-18         21.750000          108
V-05         20.580645           93
V-10         20.314286          105
V-01 had the highest average order revenue at about $22.60 per order, based on 94 orders.


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [13]:
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()

merch_share = merch_revenue / df['revenue'].sum() * 100

print(round(merch_share, 1), "%")

print("Merch made up 20.8% of the total revenue.")

20.8 %
Merch made up 20.8% of the total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [14]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

original_rows = len(df)
original_revenue = df['revenue'].sum()

joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Rows before merge:", original_rows)
print("Rows after merge:", len(joined))
print("Revenue before merge:", original_revenue)
print("Revenue after merge:", joined['revenue'].sum())

unmatched = joined[joined['vendor_name'].isna()]

print("Unmatched vendor:", unmatched['vendor_id'].unique())

Rows before merge: 400
Rows after merge: 400
Revenue before merge: 8520.0
Revenue after merge: 8520.0
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18. I kept the rows because this was a left join, so V-18 stays in the report even though it does not have a vendor name in the lookup. The row count stayed at 400 and the total revenue stayed at $8,520.00, so the merge did not change the order data.


### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [15]:
pivot = pd.pivot_table(
    df,
    index='vendor_id',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

print(pivot)

category    Drink    Food   Merch  RainGear   Total
vendor_id                                          
V-01        171.0  1338.0   373.5     241.5  2124.0
V-05        298.5   882.0   489.0     244.5  1914.0
V-10        502.5  1054.5   400.5     175.5  2133.0
V-18        582.0  1018.5   508.5     240.0  2349.0
Total      1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [16]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'

print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would tell the vendors to focus more on the categories that are bringing in the most revenue. Food made 4,293.00, which was about 50.4% of total revenue, so vendors should continue making Food a major focus. Merch made 1,771.50, or about 20.8% of total revenue, so I would also look for ways to increase Merch sales. RainGear had the lowest revenue at 901.50, so vendors could also consider whether they need to adjust how much RainGear they offer next game.

b) Q3 is probably the least trustworthy because it uses average order revenue, which can hide differences between individual orders. V-01 had the highest average at about 22.60 per order, but that average came from 94 orders. V-18 had 108 orders and an average of 21.75, which is fairly close. The averages are useful, but they do not show the full range of order sizes.